# Limpieza y alistamiento de datos — Ministerio de Hacienda y Crédito Público

*(Completa aquí una breve introducción del notebook.)*

## 1. Carga e inspección inicial

*(Completa aquí la descripción de esta sección con tus palabras.)*

In [ ]:
import pandas as pd

In [ ]:
# Visualizacion de los datos (primeras 5 filas)
df = pd.read_csv("Datos_MinisterioHacienda.csv")
df.head()

In [ ]:
# Revisamos los tipos de datos y cantidad de valores por columna
df.info()

## 2. Diagnóstico de calidad

*(Completa aquí la descripción de esta sección con tus palabras.)*

In [ ]:
# diagnostico general del dataset antes de limpiar
print("Dimensiones:", df.shape)
print("\nNulos por columna:")
print(df.isna().sum())
print("\nContratos duplicados:", df["id_contrato"].duplicated().sum())
print("\nResumen de las variables numericas:")
print(df.describe())

## 3. Limpieza general

*(Completa aquí la descripción de esta sección con tus palabras.)*

In [ ]:
# quito el contrato con valor imposible (713 billones, error de digitacion)
df = df[df['valor_del_contrato'] < 1e12].copy()
# paso las 3 columnas de fecha de texto a formato fecha
for c in ['fecha_de_firma','fecha_de_inicio_del_contrato','fecha_de_fin_del_contrato']:
    df[c] = pd.to_datetime(df[c], errors='coerce')
# calculo la duracion en dias como fecha fin menos fecha inicio
df['duracion_dias'] = (df['fecha_de_fin_del_contrato'] - df['fecha_de_inicio_del_contrato']).dt.days

In [ ]:
# Quitar espacios y unificar el formato de texto en estado y modalidad
for c in ["estado_contrato", "modalidad_de_contratacion"]:
    df[c] = df[c].str.strip().str.capitalize()
print(df["estado_contrato"].unique())
print(df["modalidad_de_contratacion"].unique())

In [ ]:
# Quito espacios y unifico mayusculas en tipo de contrato
df["tipo_de_contrato"] = df["tipo_de_contrato"].str.strip().str.capitalize()
print(df["tipo_de_contrato"].value_counts(dropna=False))

## 4. Construcción de la variable de año

*(Completa aquí la descripción de esta sección con tus palabras.)*

In [ ]:
# Revisar contratos sin fecha de firma
sin_fecha = df[df["fecha_de_firma"].isna()]
print("Contratos sin fecha de firma:", len(sin_fecha))
print("Valor total de contratos sin fecha:", sin_fecha["valor_del_contrato"].sum())

In [ ]:
# Revisar si los contratos sin fecha de firma tienen fecha de inicio
print("Sin fecha de firma:", df["fecha_de_firma"].isna().sum())
print("Sin fecha de firma pero con fecha de inicio:",
    (df["fecha_de_firma"].isna() & df["fecha_de_inicio_del_contrato"].notna()).sum())
print("Sin fecha de firma ni fecha de inicio:",
    (df["fecha_de_firma"].isna() & df["fecha_de_inicio_del_contrato"].isna()).sum())

In [ ]:
sin_anio = df[df["fecha_de_firma"].isna() & df["fecha_de_inicio_del_contrato"].isna()]
print("Contratos sin fecha para asignar año:", len(sin_anio))
print("Valor total sin año:", sin_anio["valor_del_contrato"].sum())
print("Porcentaje de contratos:", len(sin_anio) / len(df) * 100)
print("Porcentaje del valor total:",
      sin_anio["valor_del_contrato"].sum() / df["valor_del_contrato"].sum() * 100)

In [ ]:
# uso fecha de firma y, si falta, la de inicio; de ahi saco el año
df["fecha_analisis"] = df["fecha_de_firma"].fillna(df["fecha_de_inicio_del_contrato"])
df["anio"] = df["fecha_analisis"].dt.year.astype("Int64")
print("Contratos sin año:", df["anio"].isna().sum())
print("Años encontrados:", sorted(df["anio"].dropna().unique()))

## 5. Dataset de análisis y variables derivadas

*(Completa aquí la descripción de esta sección con tus palabras.)*

In [ ]:
# dataset de analisis: solo los contratos que tienen año
df_analisis = df[df["anio"].notna()].copy()
print("Registros originales:", len(df))
print("Registros para análisis:", len(df_analisis))
print("Registros excluidos por falta de año:", len(df) - len(df_analisis))

In [ ]:
# Estandarizo es_pyme, es_grupo y género (espacios, mayúsculas)
for c in ["es_pyme", "es_grupo"]:
    df_analisis[c] = df_analisis[c].str.strip().str.capitalize()
df_analisis["g_nero_representante_legal"] = df_analisis["g_nero_representante_legal"].str.strip().str.title()
df_analisis.loc[
    df_analisis["g_nero_representante_legal"].isin(["No Definido", "No definido"]),
    "g_nero_representante_legal"
] = "No definido"
print(df_analisis["es_pyme"].unique(), df_analisis["es_grupo"].unique())
print(df_analisis["g_nero_representante_legal"].unique())

In [ ]:
# Marco contratos con valor_del_contrato == 0 (no permiten calcular % de ejecución)
df_analisis["valor_contrato_cero"] = df_analisis["valor_del_contrato"] == 0
print("Contratos con valor_del_contrato = 0:", df_analisis["valor_contrato_cero"].sum())

In [ ]:
# creo la bandera es_directa: 1 si la modalidad es contratacion directa, 0 si no
df_analisis['es_directa'] = df_analisis['modalidad_de_contratacion'].str.lower().str.contains('directa').astype(int)
print(df_analisis.groupby('es_directa').size())

## 6. Validación y guardado

*(Completa aquí la descripción de esta sección con tus palabras.)*

In [ ]:
# validacion final antes de guardar
print("Dimensiones finales:", df_analisis.shape)
print(df_analisis.dtypes)

# guardo el dataset limpio para que los tres lo usen en el analisis
df_analisis.to_csv("datos_limpios.csv", index=False)
print("Guardado datos_limpios.csv")